In [1]:
import pandas as pd

In [2]:
# Read in BBHI Senior tp 1 data

# First convert the date columns to strings because they are causing trouble
columns_to_string = ['birthdate', 'date_NPS_TP5']
dtype_spec = {col: str for col in columns_to_string}

sa_tp1 = pd.read_excel("/Users/rachelmorse/Documents/2023:2024/Data/BBHI Senior/bbhi senior data.xlsx", dtype=dtype_spec)

print(f"Number of participants: {len(sa_tp1)}")


FileNotFoundError: [Errno 2] No such file or directory: '/Users/rachelmorse/Documents/2023:2024/Data/BBHI Senior/bbhi senior data.xlsx'

In [ ]:
# Filter df to only include participants with all data
# NOTE that I filtered all participants with 0 in inclusion column because they are missing lifestyle or cognition data.

filtered_SA_df = sa_tp1[(sa_tp1["Inclusion"] == 1)]

# Print number of participants before dropping N/As
print(f"Number of participants before dropping N/As: {sa_tp1.shape[0]}")

# Print number of participants remaining
print(f"Number of participants remaining: {filtered_SA_df.shape[0]}")

filtered_SA_df[["ID", "Inclusion"]].head(5)

In [ ]:
# Rename variable in BBHI Senior df so they match BBHI df

filtered_SA_df = filtered_SA_df.rename(
    columns={
        "ID": "id",  # Participant ID
        "RAVLT_delayed": "w1_delayed_recall_raw",  # RAVLT delayed
        "TMTB_sec": "w1_tmt_b_raw",  # TMT B
        "Sem_fluency": "w1_sem_fluency_raw",  # Semantic fluency
        "TMTA_sec": "w1_tmt_a_raw",  # TMT A
        "Digit_span_direct": "w1_direct_digits_raw",  # Digit span forward
        "Digit_span_back": "w1_inverse_digits_raw",  # Digit span backward
        "RAVLT_total_learn": "w1_ravlt_total",  # RAVLT total
    }
)

In [ ]:
# Function to calculate exact ages
def parse_date(date_str):
    """Parse date string based on specific criteria.
    They are formatting oddly out of excel where they are both 
    American and British format so each of these formats 
    needs to be treated differently.
    
    Args:
        date_str (str): Date string from excel.
    """
    if '00:00:00' in date_str:
        # Treat as yyyy-dd-mm format
        return pd.to_datetime(date_str, format='%Y-%d-%m 00:00:00', errors='coerce')
    else:
        # Treat as dd/mm/yyyy format
        return pd.to_datetime(date_str, format='%d/%m/%Y', errors='coerce')

def preprocess_dates(df, birth_col, collect_col):
    """Convert date columns to datetime and drop rows with invalid dates.

    Args:
        df (pd.DataFrame): DataFrame containing the date columns.
        birth_col (str): Name of the birthdate column.
        collect_col (str): Name of the data collection date column.
    """
    df[birth_col] = df[birth_col].apply(parse_date)
    df[collect_col] = df[collect_col].apply(parse_date)
    df.dropna(subset=[birth_col, collect_col], inplace=True)


def calculate_age(birth_date, collection_date):
    """Calculate the exact age based on birthdate and collection date.

    Args:
        birth_date (datetime): Birthdate of the individual.
        collection_date (datetime): Date of data collection.

    Returns:
        float: Exact age of the individual at collection date.
    """
    # Calculate the difference in years
    age = collection_date.year - birth_date.year
    # Adjust age calculation to consider months and days
    age += (collection_date.month - birth_date.month) / 12.0
    age += (collection_date.day - birth_date.day) / 365.25

    return age


# Preprocess date columns in the DataFrame
preprocess_dates(filtered_SA_df, "birthdate", "date_NPS_TP5")

# Calculate and add age to DataFrame
filtered_SA_df["w1_age"] = filtered_SA_df.apply(
    lambda row: calculate_age(row["birthdate"], row["date_NPS_TP5"]), axis=1
)

# Display the first few rows to verify results
filtered_SA_df.sample(10)[["id", "NP_age", "w1_age", "birthdate", "date_NPS_TP5"]].head(10)

In [ ]:
# Filter df to only include needed columns
filtered_SA_df = filtered_SA_df[
    [
        "id",
        "w1_age",
        "YoE",
        "sex",
        "w1_delayed_recall_raw",
        "w1_tmt_b_raw",
        "w1_sem_fluency_raw",
        "w1_tmt_a_raw",
        "w1_direct_digits_raw",
        "w1_inverse_digits_raw",
        "w1_ravlt_total",
    ]
]

In [ ]:
# Read in BBHI Senior tp 2 data

sa_tp2 = pd.read_csv("/Users/rachelmorse/Documents/2023:2024/Data/BBHI Senior/bbhi senior tp2 neuropsych.csv")

print(f"Number of participants: {len(sa_tp2)}")

In [ ]:
# Filter df to only include participants with MRI data
# filtered_sa_tp2 = sa_tp2[(sa_tp2["mri_1_w2"] == 1)]
filtered_sa_tp2 = sa_tp2

# Print number of participants before dropping N/As
print(f"Number of participants before dropping N/As: {sa_tp2.shape[0]}")

# Print number of participants remaining
print(f"Number of participants remaining: {filtered_sa_tp2.shape[0]}")

filtered_sa_tp2[["record_id_np_w2", "mri_1_w2"]].head(5)

In [ ]:
# Rename variable in so they match BBHI df
filtered_sa_tp2 = filtered_sa_tp2.rename(
    columns={
        "record_id_np_w2": "id",  # Participant ID
        "animals_w2": "w2_sem_fluency_raw",  # Semantic fluency
        "ravlt_delayed_w2": "w2_delayed_recall_raw",  # RAVLT
        "tmtb_w2": "w2_tmt_b_raw",  # TMT B
        "tmta_w2": "w2_tmt_a_raw",  # TMT A
        "digits_onwards_w2": "w2_direct_digits_raw",  # Digit span forward
        "digits_backwards_w2": "w2_inverse_digits_raw",  # Digit span backward
        "ravlt_total_w2": "w2_ravlt_total",  # RAVLT total
    }
)

filtered_sa_tp2[["id", "age", "birthdate", "date_np_wave2"]].head()

In [ ]:
# Calculate the exact ages
def preprocess_dates_t2(df, birth_col, collect_col):
    """Convert date columns to datetime and correct two-digit years.

    Args:
        df (pd.DataFrame): DataFrame containing the date columns.
        birth_col (str): Name of the birthdate column.
        collect_col (str): Name of the data collection date column.
    """
    df[birth_col] = pd.to_datetime(df[birth_col], format="%m/%d/%y", errors="coerce")
    df[collect_col] = pd.to_datetime(df[collect_col], format="%m/%d/%y", errors="coerce")

    # Adjust birthdates to 1900s if they were considered 2000s
    df[birth_col] = df[birth_col].apply(lambda x: x.replace(year=x.year - 100) if x.year >= 2000 else x)


# Call the function
preprocess_dates_t2(filtered_sa_tp2, "birthdate", "date_np_wave2")

# Calculate and add age to DataFrame
filtered_sa_tp2["w2_age"] = filtered_sa_tp2.apply(
    lambda row: calculate_age(row["birthdate"], row["date_np_wave2"]), axis=1
)

# Display the first few rows to verify results
filtered_sa_tp2[["id", "age", "w2_age", "birthdate", "date_np_wave2"]].head()

# Check output for participant with id sub-2003
filtered_sa_tp2[filtered_sa_tp2["id"] == 2003][["id", "age", "w2_age", "birthdate", "date_np_wave2"]]


In [ ]:
# Select only the required columns
filtered_sa_tp2 = filtered_sa_tp2[
    [
        "id",
        "w2_age",
        "w2_delayed_recall_raw",
        "w2_sem_fluency_raw",
        "w2_tmt_b_raw",
        "w2_tmt_a_raw",
        "w2_direct_digits_raw",
        "w2_inverse_digits_raw",
        "w2_ravlt_total",
    ]
]

In [ ]:
# Merge dfs by participant ID for tp1 and tp2
merged_df = pd.merge(filtered_SA_df, filtered_sa_tp2, on="id", how="inner")

# Print the number of participants after filtering
print(f"Number of participants: {len(merged_df)}")

merged_df.head()

In [ ]:
# Drop participants with missing neuropsych data and -1 values in the specific columns

# Get the number of rows before dropping participants
rows_before = merged_df.shape[0]

columns = [
    "YoE",  # Years of education
    "w1_age",  # Age
    "w2_age",  # Age tp2
    "w1_delayed_recall_raw",  # RAVLT tp1
    "w1_sem_fluency_raw",  # Semantic fluency tp1
    "w1_tmt_b_raw",  # TMT B tp1
    "w1_tmt_a_raw",  # TMT A tp1
    "w1_direct_digits_raw",  # Digit span forward tp1
    "w1_inverse_digits_raw",  # Digit span backward tp1
    "w2_delayed_recall_raw",  # RAVLT tp2
    "w2_sem_fluency_raw",  # Semantic fluency tp2
    "w2_tmt_b_raw",  # TMT B tp2
    "w2_tmt_a_raw",  # TMT A tp2
    "w2_direct_digits_raw",  # Digit span forward tp2
    "w2_inverse_digits_raw",  # Digit span backward tp2
    "w1_ravlt_total",  # RAVLT total tp1
    "w2_ravlt_total",  # RAVLT total tp2
]

# Drop participants with missing neuropsych data
merged_df.dropna(subset=columns, inplace=True)

# Drop participants with -1 values in specific columns
for col in columns:
    merged_df = merged_df[merged_df[col] != -1]

# Drop participant with 0 as their age
merged_df = merged_df[merged_df["w1_age"] != 0]

# Get the number of rows after dropping participants
rows_after = merged_df.shape[0]

# Calculate the number of participants dropped
participants_dropped = rows_before - rows_after

participants_dropped

In [ ]:
# Replace 1 with 'male' and 2 with 'female' in the 'sex' column
merged_df["sex"] = merged_df["sex"].replace({1: "male", 2: "female"})

merged_df[["id", "sex"]].head()

In [ ]:
# Look at age data for all available cross-sectional data (merged cohort data)

participants = merged_df["w1_age"].count()
mean_age = merged_df["w1_age"].mean()
std_dev_age = merged_df["w1_age"].std()
min_age = merged_df["w1_age"].min()
max_age = merged_df["w1_age"].max()

print(f"Number of participants: {participants}")
print(f"Mean age: {mean_age:.2f}")
print(f"SD age: {std_dev_age:.2f}")
print(f"Range age: {min_age:.2f} - {max_age:.2f}")

In [ ]:
# Export this df to a csv to use for future analysis

merged_df.to_csv("/Users/rachelmorse/Documents/2023:2024/Data/Exported data/clean_bbhi_senior.csv", index=False)